# Carvana: inventory, asking prices, and one vehicle's evidence

**Run All reads retained local evidence. It does not collect or save anything.**

Reading guide: **population, dates and completeness -> inventory/native fields ->
comparable VIN and asking-price changes -> pricing and days since first observed ->
one retained source traced through SQLite**. Settings are all in the next section.
Optional manual checks, recording instructions and synthetic examples follow the main analysis.

```text
run_carvana_daily.py --live (separate, explicit action)
  -> search pages -> retained JSON -> SQLite history + selected cycle register
  -> this notebook: coverage -> retailer/VIN join -> prices -> source evidence
```

[Daily guide](../docs/daily_inventory.md) | [Failure/recovery guide](../docs/collector_storage_handoff.md)
| [How the POST works](10_carvana_inventory.ipynb)
| [Notebook 22: native website-status checks](22_carvana_sale_status_validation.ipynb)

| Column / concept | Meaning |
| --- | --- |
| `retailer`, `vin`, `listing_id` | Retailer, physical vehicle, and its listing; retain all three |
| `asking_price_usd` | Advertised USD price, not the transaction price |
| `purchase_pending` | Native search flag; missing is unknown, false does not prove purchasability |
| `observed_at_utc` | When the source was observed locally, not a sale date |
| `available_at` | When the retained cycle/check was available to this analysis |
| `cycle_date` | Declared local date; source clocks are UTC and mileage is miles |
| `source_path` | Retained capture projection; not necessarily the original response body |

Missing from a search, pending, unavailable, and Sold-labelled are distinct observations.
A website Sold label does not establish a completed economic sale or its exact date.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / 'vehicle/src').is_dir():
    ROOT = ROOT / 'vehicle'
elif ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

## Settings and reading inputs

Edit the settings cell below, then Run All. The settings file names the plan, daily SQLite database and cycle register. The
register selects the retained cycles to analyze. We choose these inputs **once**.
`AS_OF` is an evidence cutoff: later available evidence is excluded. For a historical
view, set `AS_OF_OVERRIDE` to a timezone-aware time before running this notebook.

The optional `*_OVERRIDE` inputs select explicit retained examples/tests. An explicit
cycle selection starts with empty page checks/reviews unless those are also supplied,
so observations from different datasets cannot be silently mixed.


In [ ]:
from vehicle_tracker.daily import tracking_settings, registered_cycles, review_inputs, daily_tables
from vehicle_tracker.cycles import read_cycle_history
from vehicle_tracker.checks import CHECK_COLUMNS, read_records, select_reviews, validate_checks
from vehicle_tracker.sales import REVIEW_COLUMNS

TRACKING_CONFIG_PATH = Path(globals().get('TRACKING_CONFIG_OVERRIDE', ROOT / 'config/carvana_daily_tracking.json'))
AS_OF = globals().get('AS_OF_OVERRIDE', pd.Timestamp.now(tz='UTC').isoformat())
TRACKING_AS_OF = AS_OF  # The manual recording instructions use this same cutoff.
# Optional selection/learning controls; blank identity never prepares a saved check.
RUN_TRACKING_VIEW = globals().get('RUN_TRACKING_VIEW_OVERRIDE', True)
VIN_KEY_SETTING = globals().get('VIN_KEY_OVERRIDE')
SELECTED_IDENTITY = globals().get('SELECTED_IDENTITY_OVERRIDE', dict(retailer='', vin='', listing_id=''))
CHECK_DRAFT = globals().get('CHECK_DRAFT_OVERRIDE', dict(
    observed_status='', native_text='', checked_at='', source='', reviewer='', note=''))
OBSERVED_AGE_BINS = [0, 1, 3, 7, 14, 30, float('inf')]
OBSERVED_AGE_LABELS = ['<1 day', '1 to <3 days', '3 to <7 days',
                       '7 to <14 days', '14 to <30 days', '30+ days']
tracking = tracking_settings(TRACKING_CONFIG_PATH)

if 'CYCLE_REPORTS_OVERRIDE' in globals():
    CYCLE_REPORTS = list(CYCLE_REPORTS_OVERRIDE)
    DAILY_DATABASE = Path(globals().get('DAILY_DATABASE_OVERRIDE', globals().get('DATABASE_OVERRIDE', tracking['database'])))
    check_history_input = pd.DataFrame(columns=CHECK_COLUMNS)
    review_history = pd.DataFrame(columns=REVIEW_COLUMNS)
else:
    CYCLE_REPORTS = [entry['path'] for entry in registered_cycles(tracking)]
    DAILY_DATABASE = tracking['database']
    check_history_input, review_history = review_inputs(tracking)

if 'CHECKS_OVERRIDE' in globals():
    check_history_input = pd.DataFrame(CHECKS_OVERRIDE, columns=CHECK_COLUMNS)
if 'SALES_REVIEWS_OVERRIDE' in globals():
    review_history = pd.DataFrame(SALES_REVIEWS_OVERRIDE, columns=REVIEW_COLUMNS)

print('Settings:', TRACKING_CONFIG_PATH)
print('SQLite history:', DAILY_DATABASE)
print('Selected cycle reports:', len(CYCLE_REPORTS), '| Evidence cutoff:', AS_OF)
print('Source hashes and coverage are verified before these observations are used.')


## 1. Population, observation dates and collection completeness

`read_cycle_history` checks the retained reports and reads SQLite without writes.
The two returned tables are `daily_cycles` (one row per selected cycle) and
`daily_observations` (one row per vehicle observation). A repeated import is not
another observation. The code below exposes durations and hours between starts.

This is current-code reanalysis of retained evidence, not an old software/database
vintage. Consecutive local dates do not imply a 24-hour interval or an instantaneous
census. The later identity/coverage checks can still withhold comparisons.


In [ ]:
from vehicle_tracker.events import vin_events, daily_counts

daily_cycles = pd.DataFrame()
daily_observations = pd.DataFrame()
if CYCLE_REPORTS:
    daily_cycles, daily_observations = read_cycle_history(
        [Path(path) for path in CYCLE_REPORTS], DAILY_DATABASE, as_of=AS_OF)

daily_source_rows = daily_events = daily_summary = daily_timeline = pd.DataFrame()
daily_identity_join = collection_windows = pd.DataFrame()
daily_analysis_error = None
if daily_cycles.empty:
    print('NO DAILY CYCLES: none available at this cutoff. No counts are assumed.')
else:
    daily_cycles = daily_cycles.sort_values('cycle_date').reset_index(drop=True)
    collection_windows = daily_cycles[['cycle_id', 'cycle_date', 'timezone', 'coverage_complete', 'coverage_reason']].copy()
    for field in ['window_start', 'window_end']:
        collection_windows[field + '_local'] = [
            pd.Timestamp(getattr(cycle, field)).tz_convert(cycle.timezone).isoformat()
            for cycle in daily_cycles.itertuples()]
    starts = pd.to_datetime(daily_cycles.window_start, utc=True)
    ends = pd.to_datetime(daily_cycles.window_end, utc=True)
    collection_windows['sweep_seconds'] = (ends - starts).dt.total_seconds()
    collection_windows['hours_since_previous_start'] = starts.diff().dt.total_seconds() / 3600
    display(collection_windows)
    print('Scope ID:', daily_cycles.scope_id.unique().tolist())
    try:
        # Reuse the existing identity, scope and timeline checks; do not deduplicate errors away.
        daily_events = vin_events(daily_cycles, daily_observations, absence_days=3)
        daily_summary = daily_counts(daily_cycles, daily_events)
    except ValueError as error:
        daily_analysis_error = str(error)
        print('BLOCKED DAILY ANALYSIS:', daily_analysis_error)

### Collection health: retained work awaiting review

The latest complete registered day below remains the inventory baseline at `AS_OF`.
This separate operational check reads **current persisted checkpoints**, even when
`AS_OF` is historical. Verified retained rows in an unregistered partial cycle are
source evidence only; they never enter this notebook's inventory, absence or sales
calculations. A stopped process may leave no final error or known interruption time.


In [ ]:
from vehicle_tracker.daily import recovery_candidates

complete_selected = daily_cycles.loc[daily_cycles.coverage_complete] if not daily_cycles.empty else pd.DataFrame()
last_complete_registered_day = complete_selected.cycle_date.max() if not complete_selected.empty else 'none at cutoff'
collection_health = pd.DataFrame()
unregistered_query_health = pd.DataFrame()
if 'CYCLE_REPORTS_OVERRIDE' in globals():
    print('Explicit cycle selection: operating-folder health discovery skipped.')
else:
    print('Last complete registered day at evidence cutoff:', last_complete_registered_day)
    health_candidates = recovery_candidates(tracking)
    if not health_candidates:
        print('No unregistered retained cycles found.')
    else:
        collection_health = pd.DataFrame([dict(record_type='registered baseline at AS_OF',
            cycle_date=last_complete_registered_day, coverage_complete=not complete_selected.empty)] +
            [dict(candidate, record_type='unregistered retained cycle') for candidate in health_candidates]).reindex(columns=[
            'record_type', 'cycle_date', 'coverage_complete', 'durable_requests', 'child_requests',
            'requests_reconciled', 'unattributed_reservations', 'request_outcome_uncertain', 'window_open',
            'import_allowed', 'live_resume_allowed', 'reason'])
        display(collection_health)
        for candidate in health_candidates:
            print('Retained cycle:', candidate['path'])
            if candidate.get('unattributed_reservations', 0):
                print('Some durable reservations cannot be assigned to a page/query; unattempted status is uncertain.')
            print('Health checked at:', candidate.get('checked_at'), '| Frozen window ends:', candidate.get('window_end'))
            if candidate.get('window_open') is False:
                print('Original window expired or clock outside window: no live continuation is available; do not extend it.')
            query_health = pd.DataFrame(candidate.get('query_health', []))
            if not query_health.empty:
                unregistered_query_health = pd.concat([
                    unregistered_query_health, query_health.assign(cycle_date=candidate['cycle_date'])], ignore_index=True)
                display(query_health[['query_id', 'status', 'verified_retained_rows', 'parsed_pages',
                    'unattempted_pages', 'uncertain_pages', 'child_requests', 'reported_total', 'reason']])
            display(pd.DataFrame(candidate.get('parent_summaries', [])))
            print('Offline action:', candidate['offline_action'])
            print(candidate.get('recovery_limit', 'Validation failed; review retained evidence.'))


### Calendar coverage, including missing dates

The existing report helper supplies the calendar and follow-up queue. The actual
VIN merge and price calculations remain visible below. The helper keeps missing
calendar days and failed/partial collections separate from observed zero inventory.


In [ ]:
tracking_tables = {}
if RUN_TRACKING_VIEW:
    selected_timezone = tracking['timezone']
    if not daily_cycles.empty:
        selected_timezone = daily_cycles.timezone.iloc[0]
    tracking_tables = daily_tables(
        daily_cycles, daily_observations, as_of=AS_OF, timezone_name=selected_timezone,
        followup_limit=tracking['followup_limit'], checks=check_history_input, reviews=review_history)
    inventory_daily = tracking_tables['daily_inventory']
    vehicle_daily = tracking_tables['vehicle_observations']
    detail_followups = tracking_tables['detail_followups']
    saved_check_evidence = tracking_tables['listing_checks']
    selected_review_evidence = tracking_tables['selected_reviews']
    operating_candidates = tracking_tables['sale_candidates']
else:
    print('Operating summary skipped; explicit cycle observations remain available below.')


### Read the actual filters and each query's outcome

These filters describe the selected cycle, not a national population. A successful
page, a reconciled query, a complete declared population, and a fresh comparable
daily collection are four different checks. Stable totals alone cannot prove
stable membership in the website's changing ranked list.

In [ ]:
query_quality = population = pd.DataFrame()
# Read the latest selected cycle's existing query evidence, never collect it.
if CYCLE_REPORTS and not daily_cycles.empty:
    from vehicle_tracker.cycles import cycle_evidence
    latest_cycle = daily_cycles.sort_values('cycle_date').iloc[-1]
    latest_id = latest_cycle.cycle_id
    selected_paths = [Path(p) for p in CYCLE_REPORTS if Path(p).is_file()]
    latest_path = next((p for p in selected_paths if json.loads(p.read_text(encoding='utf-8'))['cycle_id'] == latest_id), None)
    if latest_path is not None:
        state, coverage, attempt_reports = cycle_evidence(latest_path, as_of=AS_OF)
        print('Actual selected population:')
        population = pd.DataFrame([dict(query, location_filter=query.get('location_filter', False)) for query in state['queries']])
        display(population[['query_id', 'zip_code', 'filters', 'location_filter']])
        quality_columns = ['query_id', 'coverage_complete', 'reported_total', 'stored_rows', 'reason']
        query_quality = coverage.reindex(columns=quality_columns)
        display(query_quality)
        requests_known = []
        for report_path in attempt_reports:
            retained_report = json.loads(report_path.read_text(encoding='utf-8'))
            requests_known.append(retained_report.get('requests'))
        requests_at_cutoff = sum(requests_known) if requests_known and all(pd.notna(v) for v in requests_known) else pd.NA
        duration = pd.Timestamp(latest_cycle.window_end) - pd.Timestamp(latest_cycle.window_start)
        print('Requests in retained attempt reports:', requests_at_cutoff,
              '| Observed sweep span, seconds:', duration.total_seconds())
        print('Counts describe these retained attempts; they do not measure unattended reliability.')

### Daily review: quality first, then action

The latest registered date and the calendar date at the cutoff can differ. A blank
inventory/change count is unknown, not zero. Reappearance counts below require a
valid daily comparison; the VIN history can still show a return across a gap with
uncertain timing. Query totals describe the selected population and source context.

In [ ]:
daily_review = review_coverage = pd.DataFrame()
next_action = 'No retained daily evidence at this cutoff; preview the configured collection.'
if tracking_tables and not inventory_daily.empty:
    latest_day = inventory_daily.iloc[-1]
    latest_cycle = daily_cycles.sort_values('cycle_date').iloc[-1]
    comparison_available = pd.notna(latest_day.new_since_previous_day)
    latest_events = daily_events.loc[daily_events.cycle_id.eq(latest_cycle.cycle_id)] if not daily_events.empty else pd.DataFrame()
    reappeared = (int(latest_events.reappeared_after_absence.sum())
                 if comparison_available and not latest_events.empty else pd.NA)
    if comparison_available:
        comparison_reason = 'Two complete consecutive dates'
        next_action = 'Review selected vehicle pages and record what they show.'
    elif latest_day.coverage_status == 'missing':
        comparison_reason = 'Latest calendar date has no registered cycle'
        next_action = 'Preserve the missing day; preview the next intended collection.'
    elif latest_day.coverage_status in ['partial', 'invalid']:
        comparison_reason = 'Coverage or identity validation failed'
        next_action = 'Inspect the coverage diagnostics; preserve the partial result.'
    else:
        comparison_reason = 'Need two complete consecutive dates; baseline or previous gap/partial day'
        next_action = 'Collect the unchanged pilot on the next actual local date, at a similar time.'
    daily_review = pd.DataFrame([dict(
        latest_registered_date=latest_cycle.cycle_date, calendar_date_at_cutoff=latest_day.cycle_date,
        coverage_at_cutoff=latest_day.coverage_status, inventory_count=latest_day.inventory_count,
        native_pending=latest_day.pending_true, new_vins=latest_day.new_since_previous_day,
        absent_vins=latest_day.absent_since_previous_day, reappeared_vins=reappeared,
        daily_comparison_available=comparison_available, comparison_reason=comparison_reason,
        sales_estimate=latest_day.estimated_sales)])
    with pd.option_context('display.max_colwidth', None):
        display(daily_review.T.rename(columns={0: 'Daily review'}))
if tracking_tables:
    # Queue membership and selection are different denominators.
    states = ['first_check', 'changed_since_check', 'recheck_due', 'up_to_date']
    queue_counts = detail_followups.queue_state.value_counts() if not detail_followups.empty else pd.Series(dtype='int64')
    review_coverage = queue_counts.reindex(states, fill_value=0).rename('listings').to_frame()
    display(review_coverage)
    eligible = int(detail_followups.queue_state.ne('up_to_date').sum()) if not detail_followups.empty else 0
    selected_count = int(detail_followups.selected_for_check.sum()) if not detail_followups.empty else 0
    print('Queue listings:', len(detail_followups), '| Eligible now:', eligible,
          '| Selected:', selected_count, '| Eligible overflow:', eligible - selected_count)
    print('Targeted checks are a biased sample; these counts are not sales or an accuracy estimate.')
print('NEXT ACTION:', next_action)

## 2. Inventory counts and native availability fields

First inspect the dates and a few source rows. `observed_vins` counts vehicles
actually seen; `inventory_count` is withheld on incomplete days. Pending flags
remain native observations. The sales-estimate column remains missing.


In [ ]:
if tracking_tables and not inventory_daily.empty:
    display(inventory_daily[['cycle_date', 'coverage_status', 'observed_vins', 'inventory_count',
        'pending_true', 'pending_unknown', 'average_asking_price_usd', 'estimated_sales']])
    display(vehicle_daily[['cycle_date', 'vin', 'listing_id', 'observed_at_utc',
        'purchase_pending', 'asking_price_usd', 'source_path']].head(5))
    problems = inventory_daily.loc[inventory_daily.coverage_status.ne('complete')]
    if not problems.empty:
        display(problems[['cycle_date', 'coverage_status', 'coverage_reason', 'analysis_error']])
else:
    print('No operating daily table for this selection. No counts are assumed.')

## 3. Comparable inventory and matched-vehicle asking-price changes

The outer merge uses **retailer + VIN**, retaining both listing IDs. `left_only`
means absent after, `right_only` means newly observed in this pair, and `both`
means observed on both dates. These labels do not establish sales or purchases.

We require the last two selected cycles to be complete, consecutive and consistent
with the existing identity/scope checks. We do not skip a failed day to manufacture
a daily comparison. Unknown pending values cannot establish a pending change.


In [ ]:
daily_comparison_allowed = False
daily_change_counts = pd.DataFrame()
identity_issues = pd.DataFrame()
if not daily_observations.empty:
    duplicate_vin = daily_observations.duplicated(['cycle_id', 'retailer', 'vin'], keep=False)
    duplicate_listing = daily_observations.duplicated(['cycle_id', 'retailer', 'listing_id'], keep=False)
    binding_count = daily_observations.groupby(['retailer', 'listing_id'], dropna=False).vin.transform('nunique')
    identities = daily_observations[['retailer', 'vin', 'listing_id']]
    blank_identity = identities.astype(str).apply(lambda column: column.str.strip().eq(''))
    missing_identity = identities.isna().any(axis=1) | blank_identity.any(axis=1)
    identity_issues = daily_observations.loc[duplicate_vin | duplicate_listing | binding_count.gt(1) | missing_identity]
    print('Ambiguous identity source rows (never deduplicated into price comparisons):', len(identity_issues))
    display(identity_issues.reindex(columns=['cycle_id', 'retailer', 'vin', 'listing_id', 'asking_price_usd']))
if not daily_cycles.empty:
    if daily_analysis_error is None:
        daily_source_rows = daily_observations.merge(daily_cycles[['cycle_id', 'cycle_date', 'coverage_complete']],
            on='cycle_id', validate='many_to_one')
        VIN_KEY = VIN_KEY_SETTING if VIN_KEY_SETTING is not None else (
            tuple(daily_events[['retailer', 'vin']].iloc[0]) if not daily_events.empty else (None, None))
        daily_timeline = daily_events.loc[daily_events.retailer.eq(VIN_KEY[0]) & daily_events.vin.eq(VIN_KEY[1])]
        last_two = daily_cycles.sort_values('cycle_date').tail(2)
        daily_comparison_allowed = (len(last_two) == 2 and last_two.coverage_complete.all()
            and pd.to_datetime(last_two.cycle_date).diff().iloc[-1] == pd.Timedelta(days=1))
        if daily_comparison_allowed:
            before = daily_observations.loc[daily_observations.cycle_id.eq(last_two.iloc[0].cycle_id)]
            after = daily_observations.loc[daily_observations.cycle_id.eq(last_two.iloc[1].cycle_id)]
            fields = ['retailer', 'vin', 'listing_id', 'purchase_pending', 'asking_price_usd']
            display(before[fields].head(3))
            display(after[fields].head(3))
            daily_identity_join = before.merge(after, on=['retailer', 'vin'], how='outer',
                suffixes=('_before', '_after'), indicator=True, validate='one_to_one')
            matched = daily_identity_join['_merge'].eq('both')
            known_pending = daily_identity_join.purchase_pending_before.notna() & daily_identity_join.purchase_pending_after.notna()
            daily_identity_join['pending_changed'] = daily_identity_join.purchase_pending_before.ne(
                daily_identity_join.purchase_pending_after).astype('boolean').where(matched & known_pending)
            daily_identity_join['visible_asking_price_change_usd'] = (daily_identity_join.asking_price_usd_after
                - daily_identity_join.asking_price_usd_before).where(matched)
            daily_change_counts = daily_identity_join.groupby('_merge', observed=False).size().rename('VINs').to_frame()
            print('Compared dates:', last_two.cycle_date.tolist())
            display(last_two[['cycle_date', 'window_start', 'window_end', 'available_at']])
            print('Hours between collection starts:', (
                pd.Timestamp(last_two.iloc[1].window_start) - pd.Timestamp(last_two.iloc[0].window_start)
            ).total_seconds() / 3600)
            display(daily_change_counts)
            print('Known pending changes:', int(daily_identity_join.pending_changed.eq(True).sum()),
                  '| Matched VINs with unknown pending comparison:', int((matched & ~known_pending).sum()))
            display(daily_identity_join[['vin', 'listing_id_before', 'listing_id_after', '_merge',
                'purchase_pending_before', 'purchase_pending_after', 'pending_changed',
                'visible_asking_price_change_usd']].head(12))
        else:
            print('BLOCKED DAILY JOIN: need two complete consecutive selected dates. Baseline/gap/partial is not zero change.')


### Which matched vehicles changed asking price?

Reuse the validated outer join. Subtract previous asking price from current asking
price only for VINs observed on both dates. Missing prices stay missing. A zero
previous price has no defined percentage change. Both listing IDs remain visible.


In [ ]:
# This analysis reuses the validated retailer/VIN outer join above.
price_change_counts = asking_price_means = composition_prices = pd.DataFrame()
matched_price_rows = additions = disappearances = pd.DataFrame()
if daily_comparison_allowed:
    matched = daily_identity_join['_merge'].eq('both')
    known_prices = (daily_identity_join.asking_price_usd_before.notna()
                    & daily_identity_join.asking_price_usd_after.notna())
    matched_price_rows = daily_identity_join.loc[matched, ['retailer', 'vin',
        'listing_id_before', 'listing_id_after', 'asking_price_usd_before', 'asking_price_usd_after',
        'visible_asking_price_change_usd']].copy()
    matched_price_rows['asking_price_change_pct'] = (100 * matched_price_rows.visible_asking_price_change_usd
        / matched_price_rows.asking_price_usd_before.where(matched_price_rows.asking_price_usd_before.ne(0)))
    matched_price_rows['price_comparison'] = 'unavailable'
    delta = matched_price_rows.visible_asking_price_change_usd
    for label, mask in [('unchanged', delta.eq(0)), ('cut', delta.lt(0)), ('increase', delta.gt(0))]:
        matched_price_rows.loc[mask, 'price_comparison'] = label
    price_change_counts = matched_price_rows.price_comparison.value_counts().reindex(
        ['unchanged', 'cut', 'increase', 'unavailable'], fill_value=0).rename('matched_VINs').to_frame()
    print('Matched-VIN price categories:')
    display(price_change_counts)
    changed_price_rows = matched_price_rows.loc[matched_price_rows.price_comparison.isin(['cut', 'increase'])]
    print('Actual price cuts/increases (unknown comparisons are not changes):')
    display(changed_price_rows)
    print('All matched rows, including unavailable comparisons:')
    display(matched_price_rows.head(12))
else:
    print('Asking-price comparison unavailable: need the same validated complete date pair.')

## 4. Pricing: separate inventory mix from matched-vehicle repricing

First show the denominators. Whole-inventory means use all known asking prices on
each date. Matched means use the **same VINs with both prices known**. Additions and
disappearances are shown separately, including missing prices.


In [ ]:
if daily_comparison_allowed:
    # Use an identical known-price sample on both sides; missing is never zero.
    common_known = daily_identity_join.loc[matched & known_prices]
    price_means = []
    for label, part, field in [('whole inventory before', before, 'asking_price_usd'),
            ('whole inventory after', after, 'asking_price_usd'),
            ('matched, both prices known before', common_known, 'asking_price_usd_before'),
            ('matched, both prices known after', common_known, 'asking_price_usd_after')]:
        price_means.append(dict(sample=label, VINs=len(part), known_asking_prices=part[field].count(),
            missing_asking_prices=part[field].isna().sum(), mean_asking_price_usd=part[field].mean()))
    asking_price_means = pd.DataFrame(price_means)
    additions = daily_identity_join.loc[daily_identity_join['_merge'].eq('right_only'),
        ['retailer', 'vin', 'listing_id_after', 'asking_price_usd_after']]
    disappearances = daily_identity_join.loc[daily_identity_join['_merge'].eq('left_only'),
        ['retailer', 'vin', 'listing_id_before', 'asking_price_usd_before']]
    composition_rows = []
    for label, part, field in [('additions', additions, 'asking_price_usd_after'),
                             ('disappearances', disappearances, 'asking_price_usd_before')]:
        prices = part[field]
        composition_rows.append(dict(group=label, VINs=len(part), known_asking_prices=prices.count(),
            missing_asking_prices=prices.isna().sum(), mean_asking_price_usd=prices.mean(),
            min_asking_price_usd=prices.min(), median_asking_price_usd=prices.median(),
            max_asking_price_usd=prices.max()))
    composition_prices = pd.DataFrame(composition_rows)
    display(asking_price_means)
    display(composition_prices)
    print('Added VINs:')
    display(additions.head(12))
    print('Disappeared VINs (not confirmed sales):')
    display(disappearances.head(12))
    pair_windows = collection_windows.loc[collection_windows.cycle_id.isin(last_two.cycle_id)]
    interval_hours = pair_windows.hours_since_previous_start.iloc[-1]
    print(f'Snapshot starts were {interval_hours:.4f} hours apart. This is not a measured 24-hour flow.')

### A transparent, three-step accounting breakdown

Follow this path: **previous inventory mean -> common VINs at previous prices ->
common VINs at current prices -> current inventory mean**. Each step is a simple
subtraction. The three changes add exactly to the observed overall mean change.

When every price is known, the first and last steps reflect removing disappeared
vehicles and adding new vehicles. Their sum is the composition contribution **under
this stated sequence**; the middle step is repricing the common sample. This is an
accounting description, not a causal estimate; another weighting order can allocate
contributions differently.

If prices are missing, the outer steps also include changes in which vehicles have
known prices, so we label the residual **composition plus price coverage**, not pure
inventory composition. Without a common priced VIN, the breakdown is unavailable.

In [ ]:
price_bridge = price_change_breakdown = pd.DataFrame()
if daily_comparison_allowed and not common_known.empty:
    previous_inventory_mean = before.asking_price_usd.mean()
    previous_common_mean = common_known.asking_price_usd_before.mean()
    current_common_mean = common_known.asking_price_usd_after.mean()
    current_inventory_mean = after.asking_price_usd.mean()

    remove_effect = previous_common_mean - previous_inventory_mean
    repricing_effect = current_common_mean - previous_common_mean
    add_effect = current_inventory_mean - current_common_mean
    total_mean_change = current_inventory_mean - previous_inventory_mean
    all_prices_known = before.asking_price_usd.notna().all() and after.asking_price_usd.notna().all()
    composition_label = 'inventory composition' if all_prices_known else 'composition plus price coverage'

    price_bridge = pd.DataFrame([
        dict(step='Previous inventory', mean_asking_price_usd=previous_inventory_mean, change_usd=pd.NA),
        dict(step='Common VINs, previous prices', mean_asking_price_usd=previous_common_mean, change_usd=remove_effect),
        dict(step='Common VINs, current prices', mean_asking_price_usd=current_common_mean, change_usd=repricing_effect),
        dict(step='Current inventory', mean_asking_price_usd=current_inventory_mean, change_usd=add_effect),
    ])
    price_change_breakdown = pd.DataFrame([
        dict(component='Common-sample repricing', change_usd=repricing_effect),
        dict(component=composition_label, change_usd=remove_effect + add_effect),
        dict(component='Total observed mean change', change_usd=total_mean_change),
    ])
    reconciliation_difference = total_mean_change - (remove_effect + repricing_effect + add_effect)
    display(price_bridge)
    display(price_change_breakdown)
    print('Reconciliation difference, USD (rounding only):', reconciliation_difference)
    print('Common priced VINs:', len(common_known), '| All inventory prices known:', all_prices_known)
elif daily_comparison_allowed:
    print('No common VIN with both prices known: repricing/composition breakdown unavailable.')

### Price reductions by days since first observed

The denominator is **matched retailer/VINs with both asking prices known in the
same validated pair above**. No earlier pair substitutes for a gap or partial day.
The numerator counts negative asking-price changes; median reduction uses only
those reduced vehicles and reports a positive dollar amount. Zero reductions means
a zero percentage but an undefined median, not a $0 median reduction.

`days_since_first_observed = (later observation time - first observation time) / 86400`.
The first time uses only selected **complete collections**, with observation and
evidence-availability times at or before `AS_OF`. It describes our retained history,
not actual days on market: cars present in the initial collection may have been
listed earlier. Gaps do not establish continuous listing presence.

Both vehicle observation times and their elapsed hours remain visible. Age groups
use elapsed days, not rounded calendar dates. Group counts are descriptive; one
model or a short history cannot establish a broader age pattern. These are asking
prices, not transaction prices or causal evidence of weaker demand.


In [ ]:
age_price_rows = pd.DataFrame()
age_price_summary = pd.DataFrame(columns=['eligible_matched_vehicles', 'price_reductions',
    'price_reduction_pct', 'median_reduction_usd'])
age_model_breakdown = pd.DataFrame(columns=['make', 'model', 'observed_age_group', *age_price_summary.columns])
if daily_comparison_allowed:
    # The existing outer merge has already passed scope, identity and coverage checks.
    eligible = daily_identity_join['_merge'].eq('both') & (
        daily_identity_join.asking_price_usd_before.notna() & daily_identity_join.asking_price_usd_after.notna())
    columns = ['retailer', 'vin', 'listing_id_before', 'listing_id_after',
        'observed_at_utc_before', 'observed_at_utc_after', 'make_after', 'model_after',
        'cycle_id_before', 'cycle_id_after', 'capture_id_before', 'capture_id_after', 'source_path_before', 'source_path_after',
        'asking_price_usd_before', 'asking_price_usd_after', 'visible_asking_price_change_usd']
    age_price_rows = daily_identity_join.loc[eligible].reindex(columns=columns).copy()
    history_at_cutoff = daily_observations.merge(
        daily_cycles[['cycle_id', 'available_at', 'coverage_complete']], on='cycle_id', validate='many_to_one')
    history_at_cutoff['observed_time'] = pd.to_datetime(history_at_cutoff.observed_at_utc, utc=True, format='ISO8601')
    history_at_cutoff = history_at_cutoff.loc[history_at_cutoff.coverage_complete
        & history_at_cutoff.observed_time.le(pd.Timestamp(AS_OF))
        & pd.to_datetime(history_at_cutoff.available_at, utc=True, format='ISO8601').le(pd.Timestamp(AS_OF))]
    first_seen = history_at_cutoff.groupby(['retailer', 'vin'], as_index=False).agg(first_observed_at=('observed_time', 'min'))
    age_price_rows = age_price_rows.merge(first_seen, on=['retailer', 'vin'], how='left', validate='one_to_one')
    for side in ['before', 'after']:
        age_price_rows['available_at_' + side] = age_price_rows['cycle_id_' + side].map(
            daily_cycles.set_index('cycle_id').available_at)
        age_price_rows['observed_at_utc_' + side] = pd.to_datetime(
            age_price_rows['observed_at_utc_' + side], utc=True, format='ISO8601')
    age_price_rows['elapsed_observation_hours'] = (age_price_rows.observed_at_utc_after
        - age_price_rows.observed_at_utc_before).dt.total_seconds() / 3600
    age_price_rows['days_since_first_observed'] = (age_price_rows.observed_at_utc_after
        - age_price_rows.first_observed_at).dt.total_seconds() / 86400
    if (age_price_rows.first_observed_at.isna().any()
            or age_price_rows.days_since_first_observed.lt(0).any()
            or age_price_rows.elapsed_observation_hours.le(0).any()):
        raise ValueError('Eligible price rows require valid first-observed and ordered comparison times.')
    initial_cycle = daily_cycles.loc[daily_cycles.coverage_complete].sort_values('cycle_date').iloc[0].cycle_id
    initial_members = history_at_cutoff.loc[history_at_cutoff.cycle_id.eq(initial_cycle), ['retailer', 'vin']]
    age_price_rows = age_price_rows.merge(initial_members.assign(present_in_initial_collection=True),
        on=['retailer', 'vin'], how='left', validate='one_to_one')
    age_price_rows['present_in_initial_collection'] = age_price_rows.present_in_initial_collection.eq(True)
    age_price_rows['observed_age_group'] = pd.cut(age_price_rows.days_since_first_observed,
        bins=OBSERVED_AGE_BINS, labels=OBSERVED_AGE_LABELS, right=False)
    age_price_rows['price_reduced'] = age_price_rows.visible_asking_price_change_usd.lt(0)
    age_price_rows['reduction_usd'] = (-age_price_rows.visible_asking_price_change_usd).where(age_price_rows.price_reduced)
    age_price_rows = age_price_rows.rename(columns={'make_after': 'make', 'model_after': 'model'})
    age_price_rows[['make', 'model']] = age_price_rows[['make', 'model']].fillna('unknown')
    denominator = len(age_price_rows)
    reductions = int(age_price_rows.price_reduced.sum())
    if denominator:
        age_price_summary = pd.DataFrame([dict(eligible_matched_vehicles=denominator,
            price_reductions=reductions, price_reduction_pct=100 * reductions / denominator,
            median_reduction_usd=age_price_rows.reduction_usd.median() if reductions else pd.NA)])
        age_model_breakdown = age_price_rows.groupby(['make', 'model', 'observed_age_group'],
            observed=True, dropna=False).agg(eligible_matched_vehicles=('vin', 'size'),
                price_reductions=('price_reduced', 'sum'), median_reduction_usd=('reduction_usd', 'median')).reset_index()
        age_model_breakdown['price_reduction_pct'] = (100 * age_model_breakdown.price_reductions
            / age_model_breakdown.eligible_matched_vehicles)
        display(age_price_summary)
        display(age_model_breakdown)
        print('Observation clocks and age calculation; full denominator retained in age_price_rows:')
        display(age_price_rows[['retailer', 'vin', 'listing_id_before', 'listing_id_after',
            'observed_at_utc_before', 'observed_at_utc_after', 'elapsed_observation_hours',
            'first_observed_at', 'days_since_first_observed', 'present_in_initial_collection', 'reduction_usd']].head(12))
    else:
        print('No matched vehicles with both asking prices known. Reduction percentage and median are unavailable.')
        display(age_price_summary)
else:
    print('Price reductions by days since first observed: unavailable for the selected period; need the same complete consecutive pair.')
    display(age_price_summary)


## 5. One vehicle traced back to its retained source

Copy **retailer, VIN and listing ID** into `SELECTED_IDENTITY` in Settings. The timeline follows
that retailer/VIN across listing IDs. A queue row is a suggested check, not evidence
of a sale. No vehicle is silently selected for saving.

An absent-cycle row is derived from a collection window. A missing calendar date
is a gap. A corrected check is not another visit; its earlier versions remain in
the input history. All displayed evidence respects the cutoff.


In [ ]:
check_history = pd.DataFrame(columns=CHECK_COLUMNS)
if tracking_tables and not daily_source_rows.empty:
    check_history = check_history_input.copy()
    check_history = check_history.loc[pd.to_datetime(check_history.available_at, utc=True, format='ISO8601').le(pd.Timestamp(TRACKING_AS_OF))]
    check_history = validate_checks(check_history)


In [ ]:
from vehicle_tracker.timeline import vin_timeline

IDENTITY_FIELDS = ['retailer', 'vin', 'listing_id']
selected_vehicle = selected_vin_history = pd.DataFrame()
selection_error = None
selection_options = pd.DataFrame(columns=IDENTITY_FIELDS + ['listing_url'])
if daily_analysis_error is None and not daily_observations.empty:
    selection_options = daily_observations.sort_values('observed_at_utc').drop_duplicates(IDENTITY_FIELDS, keep='last')
    if tracking_tables and not detail_followups.empty:
        print('Start with selected queue rows; older listings and controls remain in selection_options.')
        display(detail_followups.loc[detail_followups.selected_for_check,
            IDENTITY_FIELDS + ['followup_reason', 'listing_url']].head(20))
    else:
        print('No actionable queue. These are retained inventory controls, not sale candidates.')
    display(selection_options[IDENTITY_FIELDS + ['listing_url']].head(10))
    if any(SELECTED_IDENTITY.values()):
        if set(SELECTED_IDENTITY) != set(IDENTITY_FIELDS) or not all(SELECTED_IDENTITY.values()):
            selection_error = 'Supply retailer, VIN and listing ID together.'
        else:
            match = pd.Series(True, index=selection_options.index)
            for field in IDENTITY_FIELDS:
                match &= selection_options[field].eq(SELECTED_IDENTITY[field])
            selected_vehicle = selection_options.loc[match]
            if len(selected_vehicle) != 1:
                selection_error = 'No unique retained listing matches this full identity.'
                selected_vehicle = pd.DataFrame()
    if not selected_vehicle.empty:
        display(selected_vehicle[IDENTITY_FIELDS + ['listing_url', 'observed_at_utc']])
        print('Open this original URL manually:', selected_vehicle.iloc[0].listing_url)
        selected_vin_history = vin_timeline(daily_cycles, daily_observations, check_history,
            retailer=SELECTED_IDENTITY['retailer'], vin=SELECTED_IDENTITY['vin'], as_of=AS_OF)
        history_columns = ['cycle_date', 'observed_at', 'evidence_type', 'listing_id', 'purchase_pending',
            'observed_status', 'native_text', 'derived_event', 'timing_uncertain']
        with pd.option_context('display.max_colwidth', None):
            display(selected_vin_history[history_columns])
        print('Supporting times and sources (same row numbers):')
        display(selected_vin_history[['available_at', 'derived_available_at', 'window_start', 'window_end',
            'last_known_listing_id', 'source', 'correction_versions']])
    elif selection_error:
        print('SELECTION:', selection_error)
    else:
        print('Copy one full identity into SELECTED_IDENTITY, then rerun this cell.')
else:
    print('No valid retained inventory to select at this cutoff.')


### Walk one retained observation from JSON through SQLite to this calculation

If you selected a vehicle, inspect its most recent retained observation. Otherwise
this read-only example uses the first observed row; it does **not** select a vehicle
for the manual saving workflow. The SQL joins `observations` to `captures` by
`capture_id`, which is the retained projection's SHA-256. Then we open that file
and show the vehicle's source fields beside its stored values.

`price.total -> asking_price_usd` is the field mapping used by the parser. The last
table locates this same retailer/VIN in the visible before/after merge above. Older
captures retained selected projections only; their full HTTP response is unavailable.
For newer captures, `response_evidence` states whether original safe content or only
selected source JSON was retained. A hash alone cannot recreate an omitted body.

In [ ]:
import hashlib
import sqlite3

trace_observation = stored_example = source_example = evidence_trace = trace_price_comparison = pd.DataFrame()
if daily_analysis_error is None and not daily_observations.empty:
    trace_observation = daily_observations.sort_values(['observed_at_utc', 'retailer', 'vin']).head(1)
    if not selected_vehicle.empty:
        trace_observation = selected_vehicle
    trace_row = trace_observation.iloc[0]
    query = """
        SELECT o.retailer, o.vin, o.listing_id, o.observed_at_utc,
               o.asking_price_usd, o.purchase_pending, o.run_id, o.capture_id,
               c.source_path
        FROM observations AS o
        JOIN captures AS c ON o.capture_id = c.capture_id
        WHERE o.run_id = ? AND o.capture_id = ? AND o.retailer = ?
          AND o.vin = ? AND o.listing_id = ?
    """
    connection = sqlite3.connect(DAILY_DATABASE.resolve().as_uri() + '?mode=ro', uri=True)
    try:
        stored_example = pd.read_sql_query(query, connection, params=[
            trace_row.run_id, trace_row.capture_id, trace_row.retailer, trace_row.vin, trace_row.listing_id])
    finally:
        connection.close()
    if len(stored_example) != 1:
        raise ValueError('The selected observation must match exactly one stored SQLite row.')
    stored_row = stored_example.iloc[0]
    source_path = Path(stored_row.source_path)
    source_bytes = source_path.read_bytes()
    if hashlib.sha256(source_bytes).hexdigest() != trace_row.capture_id:
        raise ValueError('Retained capture changed; do not use this source for interpretation.')
    source_capture = json.loads(source_bytes)
    source_example = pd.DataFrame([
        vehicle for vehicle in source_capture['vehicles']
        if str(vehicle['vehicleId']) == trace_row.listing_id and vehicle['vin'] == trace_row.vin])
    if len(source_example) != 1:
        raise ValueError('The retained capture must contain exactly one matching listing/VIN.')
    source_vehicle = source_example.iloc[0]
    evidence_trace = pd.DataFrame([
        dict(source_field='vin', retained_value=source_vehicle.vin, sqlite_column='vin', sqlite_value=stored_row.vin),
        dict(source_field='vehicleId', retained_value=source_vehicle.vehicleId, sqlite_column='listing_id', sqlite_value=stored_row.listing_id),
        dict(source_field='price.total', retained_value=source_vehicle.price['total'], sqlite_column='asking_price_usd', sqlite_value=stored_row.asking_price_usd),
        dict(source_field='isPurchasePending', retained_value=source_vehicle.isPurchasePending, sqlite_column='purchase_pending', sqlite_value=stored_row.purchase_pending),
    ])
    print('Retained projection:', source_path)
    print('Source observation time:', source_capture['captured_at_utc'])
    print('Earlier response evidence:', source_capture.get('response_evidence', 'Legacy selected projection only; full response not retained.'))
    display(stored_example)
    display(evidence_trace)
    if not daily_identity_join.empty:
        trace_price_comparison = daily_identity_join.loc[
            daily_identity_join.retailer.eq(trace_row.retailer) & daily_identity_join.vin.eq(trace_row.vin),
            ['retailer', 'vin', 'listing_id_before', 'listing_id_after', '_merge',
             'asking_price_usd_before', 'asking_price_usd_after', 'visible_asking_price_change_usd']]
        display(trace_price_comparison)
else:
    print('No validated retained observation available for the source-to-SQL walkthrough.')

## Optional: inspect saved manual page checks

These are saved **manual browser observations**, not automatic page requests.
We matched each page's VIN and listing URL before recording the result. The source
contains the exact status wording, its location, identity evidence, and check time.

| Recorded status | What it establishes |
| --- | --- |
| `available` | The loaded page offered purchase, with a Get Started control |
| `pending` | The page displayed a purchase-in-progress or hold state |
| `sold_label` | An explicit status said this listing was sold; delivery date remains unknown |
| `unavailable` | No longer available, without an explicit sold status |
| `access_blocked` | The page could not be accessed; vehicle status is unestablished |
| `unknown` | No other category was established; read the native wording and note |

The September 8 study selected the first two pending VINs and first three
non-pending VINs in VIN order: five controls from one baseline, no missing-VIN
cohort yet. Two pre-order pages are retained as `unknown`, with “Pre-order now”
and “Inspection in progress” preserved; we did not add a new status system.

All five pages contained generic wording about equipment “as originally sold.”
That is not the vehicle's sale status. No explicit Sold status was observed.
This convenience sample cannot estimate accuracy, sales, or reappearance rates.

The merge uses retailer + VIN + listing ID. It attaches the latest inventory
observation at or before each physical page check. Different source times and
browser context can explain differences; they are not proven orders/cancellations.

In [ ]:
check_context = page_check_counts = check_history_visible = later_inventory = pd.DataFrame()
if tracking_tables and not daily_source_rows.empty:
    # Keep the latest known correction to each physical check; retain separate check times.
    check_history_visible = check_history.sort_values('available_at').drop_duplicates(
        ['retailer', 'vin', 'listing_id', 'checked_at'], keep='last')
    if saved_check_evidence.empty:
        print('No saved page checks at this cutoff. That does not mean zero sales.')
    else:
        identity = ['retailer', 'vin', 'listing_id']
        inventory_context = daily_source_rows[identity + ['cycle_date', 'observed_at_utc', 'purchase_pending']]
        check_context = saved_check_evidence.merge(inventory_context, on=identity, validate='one_to_many')
        check_context = check_context.loc[pd.to_datetime(check_context.observed_at_utc, utc=True, format='ISO8601').le(
            pd.to_datetime(check_context.checked_at, utc=True, format='ISO8601'))]
        check_context = check_context.sort_values('observed_at_utc').drop_duplicates('check_id', keep='last')
        with pd.option_context('display.max_colwidth', None):
            display(check_context[['vin', 'listing_id', 'observed_at_utc', 'purchase_pending',
                'checked_at', 'observed_status', 'native_text']])
        page_check_counts = saved_check_evidence.groupby('observed_status').size().rename('checked_listings').to_frame()
        display(page_check_counts)
        print('Counts cover checked listings only, not daily sales or the full population.')
        display(check_context[['listing_id', 'available_at', 'source', 'note']])
        repeated = check_history_visible.duplicated(identity, keep=False)
        if repeated.any():
            display(check_history_visible.loc[repeated, identity + ['checked_at', 'observed_status', 'native_text']]
                .sort_values(identity + ['checked_at']))
        else:
            print('No repeated physical page checks yet; no page-status trajectory can be measured.')
        # Same VIN can return under a new listing ID. Keep the two IDs distinct.
        later_inventory = saved_check_evidence[identity + ['checked_at']].rename(columns={'listing_id': 'checked_listing_id'}).merge(
            daily_events, on=['retailer', 'vin'], validate='many_to_many')
        later_inventory = later_inventory.loc[pd.to_datetime(later_inventory.window_start, utc=True, format='ISO8601').gt(
            pd.to_datetime(later_inventory.checked_at, utc=True, format='ISO8601'))]
        if later_inventory.empty:
            print('No later inventory cycle after these checks yet; reappearance follow-up is unavailable.')
        else:
            display(later_inventory[['vin', 'checked_listing_id', 'checked_at', 'cycle_date',
                'listing_id', 'event_type', 'observed_in_cycle', 'reappeared_after_absence', 'coverage_complete']].head(20))
elif tracking_tables:
    print('No valid inventory evidence for page-check comparison at this cutoff.')


### Candidates and the next page checks

The existing queue retains first absences, native changes and unresolved checks;
at most 20 are selected. It does not browse automatically. A recently checked
listing can be `up_to_date`; unresolved checks become due after 48 hours.
That is an operating interval, not a sales rule.

The existing three-day candidate rule is unchanged. `reviewed_sales_with_known_date`
counts only explicit dated analyst confirmations, not all sales. A saved Sold
label does not automatically create a confirmation or transaction date. Old checks
and reviews remain in their histories; the report uses only evidence available
by its cutoff. See [the recording guide](../docs/listing_checks.md) for commands.

In [ ]:
from vehicle_tracker.sales import sale_candidates
SALES_REVIEWS = select_reviews(review_history, as_of=AS_OF)
sale_candidate_rows = sale_daily_report = pd.DataFrame()
if globals().get('daily_analysis_error') is None:
    sale_candidate_rows, sale_daily_report = sale_candidates(
        daily_cycles, daily_observations, as_of=AS_OF, absence_days=3, reviews=SALES_REVIEWS)
    if not sale_candidate_rows.empty:
        display(sale_candidate_rows[['vin', 'listing_id', 'first_absent_date', 'detected_date',
            'followup_state', 'review_outcome', 'sale_date', 'review_needs_followup']])
    else:
        print('No qualifying reviewed candidate to display. Sales remain unknown.')
if globals().get('tracking_tables') and not detail_followups.empty:
    display(detail_followups[['vin', 'listing_id', 'followup_reason', 'detail_status',
        'queue_state', 'selected_for_check', 'listing_url']].head(20))

### Prepare, preview, and intentionally save a page check

Open the selected URL yourself, verify the page VIN/listing, and retain the relevant
evidence. Fill in `CHECK_DRAFT` in Settings, then rerun the notebook to
enable the manual preparation step for the selected vehicle. `checked_at` is when you actually read the page,
with a timezone, for example `2026-09-10T21:50:00-04:00`. Use the real time, not
the example. `source` points to your retained evidence, not a search snippet.

Valid statuses: `available`, `pending`, `sold_label`, `unavailable`, `access_blocked`,
`unknown`. The table above explains them. Only choose `sold_label` for an explicit
listing-specific Sold status. Exact native wording and your uncertainty matter.
Preparation validates structure and identity; it cannot authenticate your evidence.


In [ ]:
from vehicle_tracker.daily import prepare_check, record_evidence

# This cell only displays an unsaved draft. Blank defaults are not observations.
display(pd.DataFrame([CHECK_DRAFT]))
manual_recording_allowed = (bool(tracking_tables) and not selected_vehicle.empty
    and not any(name in globals() for name in ['CYCLE_REPORTS_OVERRIDE', 'DAILY_DATABASE_OVERRIDE',
        'DATABASE_OVERRIDE', 'CHECKS_OVERRIDE', 'SALES_REVIEWS_OVERRIDE', 'SELECTED_IDENTITY_OVERRIDE',
        'CHECK_DRAFT_OVERRIDE']))
if tracking_tables:
    print('Intended check-history destination:', tracking['checks'])
print('No record prepared or saved. Manual recording enabled for this selection:', manual_recording_allowed)


**Manual step 1: prepare and preview.** Use a console attached to this notebook's
kernel (in JupyterLab, right-click the notebook tab and choose **New Console for
Notebook**). Run the following lines there. Keep them out of executable notebook
cells so Run All stays read-only. Preparation makes no writes.

```python
assert manual_recording_allowed, "Select a real retained listing in the operating view first."
prepared_check = prepare_check(
    daily_observations, **SELECTED_IDENTITY, draft=CHECK_DRAFT,
    available_at=pd.Timestamp.now(tz='UTC').isoformat())
display(pd.DataFrame([prepared_check]))
print("Will append to:", tracking['checks'])
```

`available_at` is when this recorded interpretation became available. Preparation
sets it once and gives the record a stable ID; it does not change `checked_at`.
Review the identity, wording, timestamps and destination in the preview.

**Manual step 2: save that exact prepared record**, then read the stored version:

```python
assert manual_recording_allowed
changed = record_evidence(tracking, prepared_check, kind='check')
print("Recorded" if changed else "Already recorded; unchanged")
recorded_checks = read_records(tracking['checks'], CHECK_COLUMNS)
display(recorded_checks.loc[recorded_checks.check_id.eq(prepared_check['check_id'])])
```

This appends to the configured check-history CSV and uses the existing local
`cycle.lock` file. It does not collect inventory,
rewrite SQLite, or export tables. Repeating step 2 with the same `prepared_check`
does not duplicate it. Do not rerun preparation merely to retry saving. For a real
correction, keep the original physical check time, edit the wording/note, then
prepare a later version; the original remains. A new visit has a new `checked_at`.

Rerun the notebook to reread evidence. A historical `AS_OF_OVERRIDE` deliberately
excludes later checks; advance/remove that override when you want the current view.
Synthetic and alternative-cycle examples are for analysis only, never this save path.

## What happens next?

Use the opening review and the selected VIN's evidence to decide the next action.
The software supports inventory measurement and investigation. **Daily sales
accuracy, transaction timing, and reliability of a Sold label remain unvalidated.**
A targeted page sample cannot measure missed sales across the population.

Collect the unchanged pilot once per actual local date, at approximately the same
time. Preserve missed dates and failed/partial collections. Then review first
disappearances, native changes, due rechecks and a few unchanged controls. A three-day
absence creates a candidate; a Sold label starts further review, not a dated sale.

The [daily operating guide](../docs/daily_inventory.md) gives the collection and
recovery commands. [The original five-page study](../docs/status_validation_20260908.md)
documents what was observed on September 8; it is historical evidence, not today's
status. [The recording guide](../docs/listing_checks.md) explains corrections and
analyst reviews. Ignored raw evidence and databases need a local backup in addition
to the code commit.
For the frozen original and extension cohorts, open
[Notebook 22: Sold-status validation](22_carvana_sale_status_validation.ipynb).
Its experimental projections, coverage, and manual imports are separate from the
canonical checks and analyst review workflow here.

## Optional learning example: one invented VIN over eight days

Everything in this section is **synthetic and in memory only**. It never enters the
real selection or recording workflow. Read the inputs before the derived tables.

| Day | Invented evidence | Interpretation |
| --- | --- | --- |
| 1 | Listed, $20,000, not pending | First inventory observation |
| 2 | New listing ID, $19,500, pending; page says purchase in progress | Native change and page evidence |
| 3 | No collection | Gap, not absence evidence |
| 4 | Incomplete collection | Absence cannot be assessed |
| 5 | Complete collection without VIN | First assessable absence |
| 6 | Still absent; page explicitly says Sold | Website wording, not a transaction date |
| 7 | Third complete absent day | One review candidate |
| 8 | VIN returns under a third listing ID | Reappearance; revisit earlier interpretation |

The day-6 page check is entered on day 7. A day-6 cutoff must exclude it despite
its physical check time. Later evidence cannot rewrite what was known earlier.

In [ ]:
print('SYNTHETIC IN-MEMORY EXAMPLE ONLY: invented dates and VIN; no collection or sales evidence.')
synthetic_days = ['01', '02', '04', '05', '06', '07', '08']
synthetic_cycles = pd.DataFrame([dict(cycle_id='synthetic-' + day, cycle_date='2026-09-' + day, timezone='UTC', scope_id='synthetic-only',
    window_start=f'2026-09-{day}T10:00:00Z', window_end=f'2026-09-{day}T11:00:00Z', available_at=f'2026-09-{day}T11:05:00Z',
    coverage_complete=day != '04', coverage_reason='Synthetic incomplete day' if day == '04' else 'Synthetic complete day') for day in synthetic_days])
synthetic_observations = pd.DataFrame([dict(cycle_id='synthetic-' + day, retailer='carvana', vin='SYNTHETIC-VIN-A', listing_id=listing,
    observed_at_utc=f'2026-09-{day}T10:30:00Z', capture_id='synthetic-capture-' + day, run_id='synthetic-run-' + day,
    source_url='synthetic://example/' + day, listing_url='synthetic://listing/' + listing,
    asking_price_usd=price, purchase_pending=pending, vehicle_lock_type=0)
    for day, listing, price, pending in [('01', 'old', 20000, False), ('02', 'new', 19500, True), ('08', 'returned', 19500, False)]])
synthetic_events = vin_events(synthetic_cycles, synthetic_observations, absence_days=3)
synthetic_summary = daily_counts(synthetic_cycles, synthetic_events)
display(synthetic_cycles[['cycle_date', 'coverage_complete', 'coverage_reason']])
display(synthetic_observations[['cycle_id', 'vin', 'listing_id', 'asking_price_usd', 'purchase_pending']])
display(synthetic_events[['cycle_date', 'vin', 'listing_id', 'event_type', 'observed_in_cycle', 'absence_streak', 'timing_uncertain', 'asking_price_change_usd', 'estimated_sales']])
display(synthetic_summary[['cycle_date', 'skipped_days_before', 'pending_started', 'first_absence', 'persistent_absence', 'estimated_sales']])

### The same invented VIN becomes a candidate, then reappears

At the end of synthetic day 7, the three-day absence rule has produced one candidate. At the end of day 8, the same candidate is marked `reappeared`. Its original detection date does not change, no second candidate is added, and no sale is inferred. The two tables show exactly what was knowable at those different cutoffs.

In [ ]:
print('SYNTHETIC SALE-CANDIDATE EXAMPLE ONLY; no reviewed outcome or sale is invented.')
example_candidates_before, _ = sale_candidates(synthetic_cycles, synthetic_observations,
    as_of='2026-09-07T23:00:00Z', absence_days=3)
example_candidates_after, example_candidate_daily = sale_candidates(synthetic_cycles, synthetic_observations,
    as_of='2026-09-08T23:00:00Z', absence_days=3)
for label, frame in [('Known on day 7', example_candidates_before), ('Known on day 8', example_candidates_after)]:
    print(label)
    display(frame[['vin', 'detected_date', 'detected_available_at', 'followup_state',
                   'reappeared_date', 'review_outcome', 'sale_date']])
from vehicle_tracker.timeline import vin_timeline
synthetic_checks = pd.DataFrame([
    dict(check_id='synthetic-pending', retailer='carvana', vin='SYNTHETIC-VIN-A', listing_id='new',
         checked_at='2026-09-02T12:00:00Z', available_at='2026-09-02T12:05:00Z',
         observed_status='pending', native_text='Purchase in progress', source='synthetic://page/day2',
         reviewer='Invented example', note='Invented listing-specific text, not real evidence'),
    dict(check_id='synthetic-sold', retailer='carvana', vin='SYNTHETIC-VIN-A', listing_id='new',
         checked_at='2026-09-06T12:00:00Z', available_at='2026-09-07T12:05:00Z',
         observed_status='sold_label', native_text='Sold', source='synthetic://page/day6',
         reviewer='Invented example', note='Entered later; no transaction date established')])
display(synthetic_checks[['checked_at', 'available_at', 'observed_status', 'native_text', 'source']])
synthetic_history_before = vin_timeline(synthetic_cycles, synthetic_observations, synthetic_checks,
    retailer='carvana', vin='SYNTHETIC-VIN-A', as_of='2026-09-06T23:00:00Z')
synthetic_history_after = vin_timeline(synthetic_cycles, synthetic_observations, synthetic_checks,
    retailer='carvana', vin='SYNTHETIC-VIN-A', as_of='2026-09-08T23:00:00Z')
print('Known at day 6: later-entered Sold wording is excluded.')
display(synthetic_history_before[['cycle_date', 'evidence_type', 'listing_id', 'observed_status', 'native_text', 'derived_event']])
print('Known at day 8: page wording plus later inventory return; no sale date is inferred.')
display(synthetic_history_after[['cycle_date', 'evidence_type', 'listing_id', 'observed_status', 'native_text', 'derived_event', 'timing_uncertain']])
